In [1]:
from google.colab import drive, files
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0, MobileNetV2, ResNet50
from tensorflow.keras.optimizers import Adam

In [3]:
train_dir = '/content/drive/MyDrive/Deepfake_Project/Dataset_Split/extracted/train'
val_dir = '/content/drive/MyDrive/Deepfake_Project/Dataset_Split/extracted/val'
test_dir = '/content/drive/MyDrive/Deepfake_Project/Dataset_Split/extracted/test'

In [4]:
# Image Preprocessing
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1./255)

In [5]:
# Load datasets
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(128, 128),
    batch_size=32,
    class_mode='binary'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(128, 128),
    batch_size=32,
    class_mode='binary'
)

Found 11772 images belonging to 2 classes.
Found 12194 images belonging to 2 classes.


# 1. EfficientNetB0 (Input size: 128x128)

In [6]:
base_model_EfficientNetB0 = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(128, 128, 3))
base_model_EfficientNetB0.trainable = False  # Freezing base model

model_EfficientNetB0 = models.Sequential([
    base_model_EfficientNetB0,
    layers.GlobalAveragePooling2D(),
    layers.Dense(512, activation='relu'),
    layers.Dense(1, activation='sigmoid')  # Binary classification: FAKE vs REAL
])

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [7]:
model_EfficientNetB0.compile(optimizer=Adam(learning_rate=0.0001), loss='binary_crossentropy', metrics=['accuracy'])
model_EfficientNetB0.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 4, 4, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       655,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           513 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,705,956 (17.95 MB)

 Trainable params: 656,385 (2.50 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [8]:
checkpoint_callback_EfficientNetB0 = tf.keras.callbacks.ModelCheckpoint(
    'efficientnet_model.h5',
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

history_EfficientNetB0 = model_EfficientNetB0.fit(
    train_generator,
    epochs=10,
    validation_data=val_generator,
    callbacks=[checkpoint_callback_EfficientNetB0]
)

/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
 12/368 ━━━━━━━━━━━━━━━━━━━━ 1:14:01 12s/step - accuracy: 0.7414 - loss: 0.4995

KeyboardInterrupt: 

In [ ]:
from google.colab import files
files.download('efficientnet_model.h5')

In [ ]:
# Plotting the training and validation accuracies
train_accuracy = history_EfficientNetB0.history['accuracy']
val_accuracy = history_EfficientNetB0.history['val_accuracy']

print(f"Final Training Accuracy: {train_accuracy[-1]:.2f}")
print(f"Final Validation Accuracy: {val_accuracy[-1]:.2f}")

In [ ]:
import matplotlib.pyplot as plt
# Plot accuracy
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(train_accuracy, label='Training Accuracy')
plt.plot(val_accuracy, label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Plot loss
plt.subplot(1, 2, 2)
plt.plot(history_EfficientNetB0.history['loss'], label='Training Loss')
plt.plot(history_EfficientNetB0.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

# 2. MobileNetV2 (Input size: 128x128)

In [ ]:
base_model_MobileNetV2 = MobileNetV2(weights='imagenet', include_top=False, input_shape=(128, 128, 3))
base_model_MobileNetV2.trainable = False  # Freezing base model

model_MobileNetV2 = models.Sequential([
    base_model_MobileNetV2,
    layers.GlobalAveragePooling2D(),
    layers.Dense(512, activation='relu'),
    layers.Dense(1, activation='sigmoid')  # Binary classification: FAKE vs REAL
])

In [ ]:
model_MobileNetV2.compile(optimizer=Adam(learning_rate=0.0001), loss='binary_crossentropy', metrics=['accuracy'])
model_MobileNetV2.summary()

In [ ]:
# Training the model with checkpointing
checkpoint_callback_MobileNetV2 = tf.keras.callbacks.ModelCheckpoint(
    'mobilenetv2_model.h5',
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

history_MobileNetV2 = model_MobileNetV2.fit(
    train_generator,
    epochs=10,
    validation_data=val_generator,
    callbacks=[checkpoint_callback_MobileNetV2]
)

In [ ]:
from google.colab import files
files.download('mobilenetv2_model.h5')

In [ ]:
# Plotting the training and validation accuracies
train_accuracy = history_MobileNetV2.history['accuracy']
val_accuracy = history_MobileNetV2.history['val_accuracy']

print(f"Final Training Accuracy: {train_accuracy[-1]:.2f}")
print(f"Final Validation Accuracy: {val_accuracy[-1]:.2f}")

In [ ]:
import matplotlib.pyplot as plt
# Plot accuracy
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(train_accuracy, label='Training Accuracy')
plt.plot(val_accuracy, label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Plot loss
plt.subplot(1, 2, 2)
plt.plot(history_MobileNetV2['loss'], label='Training Loss')
plt.plot(history_MobileNetV2.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

# 3. ResNet50 (Input size: 128x128)

In [ ]:
base_model_ResNet50 = ResNet50(weights='imagenet', include_top=False, input_shape=(128, 128, 3))
base_model_ResNet50.trainable = False

model_ResNet50 = models.Sequential([
    base_model_ResNet50,
    layers.GlobalAveragePooling2D(),
    layers.Dense(512, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

In [ ]:
model_ResNet50.compile(optimizer=Adam(learning_rate=0.0001), loss='binary_crossentropy', metrics=['accuracy'])
model_ResNet50.summary()

In [ ]:
# Training the model with checkpointing
checkpoint_callback_ResNet50 = tf.keras.callbacks.ModelCheckpoint(
    'resnet_model.h5',
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

history_ResNet50 = model_ResNet50.fit(
    train_generator,
    epochs=10,
    validation_data=val_generator,
    callbacks=[checkpoint_callback_ResNet50]
)

In [ ]:
from google.colab import files
files.download('resnet_model.h5')

In [ ]:
# Plotting the training and validation accuracies
train_accuracy = history_ResNet50.history['accuracy']
val_accuracy = history_ResNet50.history['val_accuracy']

print(f"Final Training Accuracy: {train_accuracy[-1]:.2f}")
print(f"Final Validation Accuracy: {val_accuracy[-1]:.2f}")

In [ ]:
import matplotlib.pyplot as plt
# Plot accuracy
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(train_accuracy, label='Training Accuracy')
plt.plot(val_accuracy, label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Plot loss
plt.subplot(1, 2, 2)
plt.plot(history_ResNet50.history['loss'], label='Training Loss')
plt.plot(history_ResNet50.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()